<a href="https://colab.research.google.com/github/aniketpathak028/license-plate-recognition/blob/main/license_plate_recognition_yolo%2Bocr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 90.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 4.1 MB/s eta 0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.27.1
    Uninstalling typer-0.27.1:
      Successfully uninstalled typer-0.27.1


In [ ]:
from google.colab import userdata
from roboflow import Roboflow
rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
project = rf.workspace("roboflow-universe-projects").project("license-plate-recognition-rxg4e")
version = project.version(11)
dataset = version.download(
    "yolov8", location="/content/drive/MyDrive/license_plate_dataset"
)

loading Roboflow workspace...
loading Roboflow project...


In [ ]:
!pip install ultralytics
!pip install easyocr

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 972.1/972.1 kB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 19.8 MB/s eta 0:00:00


In [ ]:
# import libraries
from ultralytics import YOLO
import shutil, os

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


In [51]:
# define Google Drive paths
drive_model_dir = "/content/drive/MyDrive/saved_models"
best_weights_path = os.path.join(drive_model_dir, "license_plate_best.pt")

# check if weights already exist in Google Drive
if os.path.exists(best_weights_path):
    print(
        f"Weights already found at {best_weights_path}. Skipping training and loading existing model."
    )
    model = YOLO(best_weights_path)
else:
    print("No saved weights found. Starting training...")

    # load pretrained YOLOv8 nano model
    model = YOLO("yolov8n.pt")

    # finetune yolov8 on license dataset
    results = model.train(
        data=dataset.location + "/data.yaml",
        epochs=5,
        imgsz=640,
        batch=16,
        workers=2,
        device=0,
    )

    # copy weights to G-Drive folder
    os.makedirs(drive_model_dir, exist_ok=True)
    source_dir = "runs/detect/train/weights"

    # make sure source files exist before copying
    if os.path.exists(os.path.join(source_dir, "best.pt")):
        shutil.copy(
            os.path.join(source_dir, "best.pt"),
            os.path.join(drive_model_dir, "license_plate_best.pt"),
        )
        shutil.copy(
            os.path.join(source_dir, "last.pt"),
            os.path.join(drive_model_dir, "license_plate_last.pt"),
        )
        print("Weights successfully saved to Google Drive!")
    else:
        print("Training completed, but expected weight files were not found.")

    model = YOLO(best_weights_path)

Weights already found at /content/drive/MyDrive/saved_models/license_plate_best.pt. Skipping training and loading existing model.


In [52]:
import cv2
import numpy as np
from ultralytics import YOLO
import easyocr
import re
from collections import defaultdict, deque

In [53]:
# import finetuned YOLO weights and OCR model
reader = easyocr.Reader(["en"], gpu=True)

# Regex - 2 letters + 2 numbers + 3 letters
plate_pattern = re.compile(r"^[A-Z]{2}[0-9]{2}[A-Z]{3}$")

In [54]:
def correct_plate_format(ocr_text):
  mapping_num_to_alpha = {"0": "O", "1": "I", "5": "S", "8": "B"}
  mapping_alpha_to_num = {"O": "0", "I": "1", "S": "5", "B": "8"}

  ocr_text= ocr_text.upper().replace(" ", "")
  if len(ocr_text)!=7:
    return "" # discard if wrong length

  corrected = []
  for i, ch in enumerate(ocr_text):
    if i<2 or i>=4: # alphabet positions
      if ch.isdigit() and ch in mapping_num_to_alpha:
        corrected.append(mapping_num_to_alpha[ch])
      elif ch.isalpha():
        corrected.append(ch)
    else: # numeric positions
      if ch.isalpha() and ch in mapping_alpha_to_num:
        corrected.append(mapping_alpha_to_num[ch])
      elif ch.isdigit():
        corrected.append(ch)
      else:
        return "" # invalid char
  return "".join(corrected)

In [55]:
def recognize_plate(plate_crop):
  if plate_crop.size == 0:
    return ""

  # preprocess for OCR
  gray= cv2.cvtColor(plate_crop, cv2.COLOR_BGR2GRAY)
  _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
  plate_resized = cv2.resize(thresh, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)

  try:
    ocr_result = reader.readtext(
        plate_resized, detail=0, allowlist= 'ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'
    )
    if len(ocr_result) > 0:
      candidate = correct_plate_format(ocr_result[0])
      if candidate and plate_pattern.match(candidate):
        return candidate
  except:
    pass

  return ""

In [56]:
# number plate stabilization buffer

plate_history = defaultdict(lambda: deque(maxlen=10))
plate_final= {}

def get_box_id(x1, y1, x2, y2):
  # use rounded coordinates as a pseudo ID
  return f"{int(x1/10)}_{int(y1/10)}_{int(x2/10)}_{int(y2/10)}"

def get_stable_plate(box_id, new_text):
  if new_text:
    plate_history[box_id].append(new_text)
    # majority vote
    most_common = max(set(plate_history[box_id]), key= plate_history[box_id].count)
    plate_final[box_id]= most_common
  return plate_final.get(box_id, "")

In [ ]:
input_video = "vehicle_video.mp4"
cap = cv2.VideoCapture(input_video)
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
output_video = "output_with_licensev3.mp4"
out = cv2.VideoWriter(
    output_video,
    fourcc,
    cap.get(cv2.CAP_PROP_FPS),
    (
        int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
    ),
)
CONF_THRESH = 0.3

# Operating frame by frame
while cap.isOpened():
  ret, frame = cap.read()
  if not ret:
    break

  results = model(frame, verbose=False)
  for r in results:
    boxes = r.boxes
    for box in boxes:
      conf = float(box.conf.cpu().numpy())
      if conf < CONF_THRESH:
        continue

      x1, y1, x2, y2 = map(int, box.xyxy.cpu().numpy()[0])
      plate_crop = frame[y1:y2, x1:x2]
      text = recognize_plate(plate_crop)

      box_id = get_box_id(x1, y1, x2, y2)
      stable_text = get_stable_plate(box_id, text)

      cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 3)

      if plate_crop.size > 0:
        overlay_h, overlay_w = 150, 400
        plate_resized = cv2.resize(plate_crop, (overlay_w, overlay_h))
        oy1 = max(0, y1 - overlay_h - 40)
        ox1 = x1
        oy2, ox2 = oy1 + overlay_h, ox1 + overlay_w
        if oy2 <= frame.shape[0] and ox2 <= frame.shape[1]:
          frame[oy1:oy2, ox1:ox2] = plate_resized

          if stable_text:
            cv2.putText(
                frame,
                stable_text,
                (ox1, oy1 - 20),
                cv2.FONT_HERSHEY_SIMPLEX,
                2,
                (0, 0, 0),
                6,
            )
            cv2.putText(
                frame,
                stable_text,
                (ox1, oy1 - 20),
                cv2.FONT_HERSHEY_SIMPLEX,
                2,
                (255, 255, 255),
                3,
            )
  out.write(frame)

cap.release()
out.release()
print("Processing complete! Video saved as:", output_video)